In [1]:
import os, json, shutil, hashlib
import pandas as pd

os.chdir("/scratch/jq2uw/derm_vlms")

VLMS = {
    "DermatoLlama": "results/dermato_llama_predictions_paired.csv",
    "MedGemma":     "results/medgemma_predictions_paired.csv",
    "GPT-5.3":      "results/gpt53_predictions_paired.csv",
}
IMAGE_DIR     = "results/images"
INTERFACE_SRC = "res_eng/interface"
OUTPUT_DIR    = "results/interface_share"
OUTPUT_HTML   = os.path.join(OUTPUT_DIR, "index.html")
OUTPUT_IMGS   = os.path.join(OUTPUT_DIR, "images")

QUESTIONS = ["Is the AI’s assessment accurate for this case? If not, what is incorrect and what should it say instead?"]

# --- load CSV data, keep only combined images and describe_then_classify ---
vlm_data = {}
all_ids = set()
for name, path in VLMS.items():
    if not os.path.exists(path):
        print(f"[SKIP] {name}: {path} not found (predictions may still be running)")
        continue
    df = pd.read_csv(path)
    df = df[df["id"].str.endswith("_combined")]
    if df.empty:
        print(f"[SKIP] {name}: no combined rows yet")
        continue
    records = df[["id", "describe_then_classify"]].to_dict("records")
    vlm_data[name] = records
    all_ids.update(df["id"])

# --- Build image path map (relative to the HTML file) ---
images = {}
for img_id in sorted(all_ids):
    images[img_id] = f"images/{img_id}.jpg"

# --- Copy high-res images to output folder ---
os.makedirs(OUTPUT_IMGS, exist_ok=True)
copied = 0
for img_id in sorted(all_ids):
    src = os.path.join(IMAGE_DIR, f"{img_id}.jpg")
    dst = os.path.join(OUTPUT_IMGS, f"{img_id}.jpg")
    shutil.copy2(src, dst)
    copied += 1

fingerprint_src = json.dumps(QUESTIONS) + json.dumps(sorted(all_ids))
CONFIG_HASH = hashlib.md5(fingerprint_src.encode()).hexdigest()[:8]

print(f"Loaded {len(vlm_data)} VLMs, {len(all_ids)} images (config hash: {CONFIG_HASH})")
print(f"Copied {copied} images to {OUTPUT_IMGS}")

Loaded 3 VLMs, 10 images (config hash: feabceb5)
Copied 10 images to results/interface_share/images


In [2]:
import re

# ── Regex patterns to locate the start of the "diagnoses" section ──
# Ordered from most specific → most generic so the first match wins.
DIAG_PATTERNS = [
    # Markdown headings / bold headers  (MedGemma style)
    re.compile(
        r'^\s*\*{0,2}(?:Top\s+\d+\s+)?Differential\s+Diagnos[ei]s'
        r'(?:\s*\(Top\s*\d+\))?[:\s*]*\*{0,2}\s*$',
        re.IGNORECASE | re.MULTILINE,
    ),
    # Inline "Top N differential diagnoses:" on its own line  (GPT-5.3 style)
    re.compile(
        r'^(?:#*\s*)?(?:\*{0,2})Top\s+(?:\d+\s+)?differential\s+diagnos[ei]s[:\s]*(?:\*{0,2})',
        re.IGNORECASE | re.MULTILINE,
    ),
    # "Top N diagnoses:" standalone
    re.compile(
        r'^(?:#*\s*)?(?:\*{0,2})Top\s+\d+\s+diagnos[ei]s[:\s]*(?:\*{0,2})',
        re.IGNORECASE | re.MULTILINE,
    ),
    # Sentence lead-in (DermatoLlama style):
    # "Based on these characteristics, my differential diagnosis would include:"
    re.compile(
        r'(?:Based on|Given)\s+.{0,80}?(?:differential|top\s+\d+)\s+diagnos[ei]s\b',
        re.IGNORECASE,
    ),
    # "The differential diagnosis/diagnoses would include:"  /  "my differential would include:"
    re.compile(
        r'(?:the|my)\s+differential\s+(?:diagnos[ei]s\s+)?(?:would\s+)?include[s]?\s*:',
        re.IGNORECASE,
    ),
    # "The differential includes:" / "Differentials are:"
    re.compile(
        r'(?:the\s+)?(?:top\s+)?differentials?\s+(?:are|include)[s]?\s*:',
        re.IGNORECASE,
    ),
]


def split_response(text):
    """Split a VLM response into (description, diagnoses).

    Tries each DIAG_PATTERN in order. On the first match, everything
    before is `description` and the match + everything after is `diagnoses`.
    Falls back to returning (text, '') if no pattern matches.
    """
    if not text:
        return ('', '')

    for pat in DIAG_PATTERNS:
        m = pat.search(text)
        if m:
            desc = text[:m.start()].strip()
            diag = text[m.start():].strip()
            return (desc, diag)
    return (text.strip(), '')


def clean_markdown(text):
    """Strip markdown formatting, horizontal rules, and excess whitespace."""
    if not text:
        return ''
    t = text
    # bold / italic / stray asterisks
    t = re.sub(r'\*{3,}', '', t)
    t = re.sub(r'\*\*(.+?)\*\*', r'\1', t, flags=re.DOTALL)
    t = re.sub(r'(?<!\*)\*(?!\*)(.+?)(?<!\*)\*(?!\*)', r'\1', t, flags=re.DOTALL)
    t = t.replace('*', '')
    # markdown headings
    t = re.sub(r'^#{1,6}\s*', '', t, flags=re.MULTILINE)
    # horizontal rules
    t = re.sub(r'^[-_]{3,}\s*$', '', t, flags=re.MULTILINE)
    # collapse excessive blank lines
    t = re.sub(r'\n{3,}', '\n\n', t)
    return t.strip()


def split_sentences(text):
    """Split text into sentences on period-space boundaries.

    Uses findall to grab each sentence (text ending with a period)
    while keeping decimals like '0.5 cm' intact.
    """
    if not text:
        return []
    # Match: any characters ending with a period, followed by whitespace or end.
    # The period must NOT be preceded by a digit followed by a decimal digit
    # (to protect "0.5"), and we skip common header/label lines.
    parts = re.findall(r'[^.]*\.', text)
    if not parts:
        return [text.strip()] if text.strip() else []

    sentences = []
    buf = ''
    for part in parts:
        buf += part
        stripped = buf.strip()
        if not stripped:
            continue
        # Don't split if the period is part of a decimal (e.g. "0.5")
        if re.search(r'\d\.$', stripped) and re.search(r'\d\.\d', text[text.find(stripped):text.find(stripped)+len(stripped)+2]):
            continue
        # Don't split on common abbreviations
        if re.search(r'\b(?:Dr|Mr|Mrs|Ms|vs|etc|approx|e\.g|i\.e|Fig|No)\.$', stripped):
            continue
        sentences.append(stripped)
        buf = ''

    if buf.strip():
        sentences.append(buf.strip())

    return sentences


def parse_diagnoses(diag_text):
    """Extract individual numbered diagnoses from the diagnosis section text."""
    if not diag_text:
        return []

    # Try line-start numbers with period: "1. ", "2. "
    pat_dot = re.compile(r'(?:^|\n)\s*\d+\.\s+', re.MULTILINE)
    matches = list(pat_dot.finditer(diag_text))

    # Fallback: parenthesis numbering anywhere: "1) ", "2) "
    if not matches:
        pat_paren = re.compile(r'\d+\)\s*')
        matches = list(pat_paren.finditer(diag_text))

    if not matches:
        return []

    items = []
    for i, m in enumerate(matches):
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(diag_text)
        text = diag_text[start:end].strip()
        text = re.sub(r'[,;]\s*$', '', text)
        text = re.sub(r'\s+and\s*$', '', text, flags=re.IGNORECASE)
        if text:
            items.append(text)

    return items


# ── Apply splitting + cleaning + diagnosis parsing to every record ──
split_stats = {vlm: {'matched': 0, 'missed': 0} for vlm in vlm_data}

for vlm, records in vlm_data.items():
    for rec in records:
        raw = rec.get('describe_then_classify', '')
        desc_raw, diag_raw = split_response(raw)
        rec['description'] = clean_markdown(desc_raw)
        rec['diagnoses']   = clean_markdown(diag_raw)
        rec['diagnosis_list'] = parse_diagnoses(rec['diagnoses'])
        rec['sentence_list'] = split_sentences(rec['description'])
        if diag_raw:
            split_stats[vlm]['matched'] += 1
        else:
            split_stats[vlm]['missed'] += 1

print("=== Split statistics ===")
for vlm, stats in split_stats.items():
    total = stats['matched'] + stats['missed']
    print(f"  {vlm:20s}  matched={stats['matched']}/{total}  missed={stats['missed']}")

print("\n=== Diagnosis parsing ===")
for vlm, records in vlm_data.items():
    counts = [len(r['diagnosis_list']) for r in records]
    print(f"  {vlm:20s}  items per record: {counts}")

print("\n=== Sentence parsing ===")
for vlm, records in vlm_data.items():
    counts = [len(r['sentence_list']) for r in records]
    print(f"  {vlm:20s}  sentences per record: {counts}")


=== Split statistics ===
  DermatoLlama          matched=8/10  missed=2
  MedGemma              matched=10/10  missed=0
  GPT-5.3               matched=10/10  missed=0

=== Diagnosis parsing ===
  DermatoLlama          items per record: [3, 3, 3, 3, 3, 3, 0, 3, 3, 0]
  MedGemma              items per record: [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
  GPT-5.3               items per record: [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]

=== Sentence parsing ===
  DermatoLlama          sentences per record: [4, 3, 3, 3, 3, 3, 4, 4, 4, 4]
  MedGemma              sentences per record: [6, 4, 3, 5, 5, 4, 5, 7, 5, 5]
  GPT-5.3               sentences per record: [4, 4, 1, 7, 5, 1, 5, 7, 4, 4]


In [3]:
# ── Verify: show first 2 records per VLM, then show ALL missed records ──
PREVIEW_N = 2
for vlm, records in vlm_data.items():
    print(f"\n{'='*72}")
    print(f"  VLM: {vlm}")
    print(f"{'='*72}")
    for rec in records[:PREVIEW_N]:
        print(f"\n--- {rec['id']} ---")
        print(f"[DESCRIPTION] (first 300 chars):\n{rec['description'][:300]}")
        print(f"\n[DIAGNOSES] (first 300 chars):\n{rec['diagnoses'][:300]}")
        print()

print(f"\n{'='*72}")
print("  MISSED RECORDS (no diagnoses split found)")
print(f"{'='*72}")
for vlm, records in vlm_data.items():
    for rec in records:
        if not rec['diagnoses']:
            print(f"\n--- {vlm} / {rec['id']} ---")
            print(f"[FULL RESPONSE]:\n{rec['description'][:500]}")
            print()



  VLM: DermatoLlama

--- 1_combined ---
[DESCRIPTION] (first 300 chars):
The lesion presents as a pink, erythematous patch with a central area of scaling and a surrounding area of fine, white, reticular scaling. The lesion has a somewhat irregular border and appears to be slightly raised. The central area of the lesion shows a pinkish-red color with a network of fine, wh

[DIAGNOSES] (first 300 chars):
Based on these characteristics, my differential diagnosis would include: 1) Actinic Keratosis, 2) Squamous Cell Carcinoma in-situ, and 3) Bowen's Disease.


--- 2_combined ---
[DESCRIPTION] (first 300 chars):
The lesion is a well-defined, erythematous papule with a central crusted or ulcerated area. It appears to have a slightly raised border. There are also several smaller, similar lesions present on the surrounding skin.

[DIAGNOSES] (first 300 chars):
The differential diagnosis would include: 1) Actinic Keratosis, 2) Squamous Cell Carcinoma, 3) Nodular Basal Cell Carcinoma.


  VLM: 

In [4]:
# --- Read source files ---
css      = open(os.path.join(INTERFACE_SRC, "style.css")).read()
js       = open(os.path.join(INTERFACE_SRC, "app.js")).read()
template = open(os.path.join(INTERFACE_SRC, "template.html")).read()

# --- Build the data block injected before app.js ---
data_block = f"""
const VLM_DATA  = {json.dumps(vlm_data)};
const IMAGES    = {json.dumps(images)};
const QUESTIONS = {json.dumps(QUESTIONS)};
const TASK_KEY  = 'describe_then_classify';
const SK = 'vlm_eval_dc_{CONFIG_HASH}';
"""

# --- Stitch into single HTML ---
html = template
html = html.replace("__CSS__", css)
html = html.replace("__DATA__", data_block)
html = html.replace("__JS__", js)

os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(OUTPUT_HTML, "w") as f:
    f.write(html)

size_kb = os.path.getsize(OUTPUT_HTML) / 1024
print(f"Generated {OUTPUT_HTML} ({size_kb:.1f} KB)")
print(f"Output folder: {OUTPUT_DIR}/")
print(f"  index.html + images/ -> ready to zip and share")

Generated results/interface_share/index.html (137.2 KB)
Output folder: results/interface_share/
  index.html + images/ -> ready to zip and share
